## Real-Time Energy Dashboard (Spark → Redis → Jupyter)

This dashboard visualizes:

- **City-level energy usage** aggregated by Spark Structured Streaming
- Last *N* minutes of `total_watts`, `avg_watts`, and `max_watts`
- Real-time **spike / anomaly alerts**

**Make sure these are running first:**
- `producer.py`
- `stream_job.py`
- Your Docker cluster (Redpanda + Redis + Spark)

Then run the cells below.

## Installing plotly and pandas

In [87]:
# If these are already installed in the image, you can skip this cell.
%pip install --quiet plotly pandas redis python-dateutil tqdm

import plotly, pandas  # sanity check prints
print("plotly", plotly.__version__)
print("pandas", pandas.__version__)


Note: you may need to restart the kernel to use updated packages.
plotly 6.4.0
pandas 2.2.3


## Imports & Plotly Setup

In [ ]:
import os, json, time, redis, pandas as pd
from datetime import datetime, timedelta, timezone
from dateutil import tz
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Make Plotly display inside Jupyter
pio.renderers.default = "notebook_connected"


## Redis connector for in-container (spark) vs host runs

In [90]:
# One-cell Redis connector that works both:
#  - inside the Spark container (use host 'redis')
#  - on your Mac/host notebook (use host '127.0.0.1' / 'localhost')
#
# Optional env vars:
#   REDIS_HOST, REDIS_PORT, REDIS_PASSWORD
#
# Example:
#   r = connect_redis_smart()
#   r.ping()

import os, socket, time, sys
from typing import List, Optional

try:
    import redis
except ImportError:
    raise ImportError(
        "The 'redis' package is not installed in this environment.\n"
        "Inside the Spark container, run: pip install redis\n"
        "On your Mac, run: pip install redis"
    )

def _is_running_in_docker() -> bool:
    # Most robust quick checks for containerized envs
    if os.path.exists("/.dockerenv"):
        return True
    cgroup = "/proc/1/cgroup"
    try:
        if os.path.exists(cgroup):
            with open(cgroup, "rt", encoding="utf-8", errors="ignore") as f:
                data = f.read()
            if "docker" in data or "containerd" in data:
                return True
    except Exception:
        pass
    # Fallback heuristic: Spark images often set SPARK_HOME
    return bool(os.getenv("SPARK_HOME"))

def _uniq(seq: List[Optional[str]]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if not x:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def connect_redis_smart(
    host: Optional[str] = None,
    port: Optional[int] = None,
    password: Optional[str] = None,
    tries: int = 8,
    per_try_timeout: float = 1.5,
    backoff: float = 0.75,
) -> "redis.Redis":
    """
    Try multiple host candidates based on environment (Docker vs host).
    Returns a connected redis.Redis client or raises ConnectionError.
    """
    resolved_port = int(port or os.getenv("REDIS_PORT", "6379"))
    resolved_pwd = password if password is not None else os.getenv("REDIS_PASSWORD")

    in_docker = _is_running_in_docker()

    # Build candidate hosts in priority order.
    # If user explicitly set REDIS_HOST, try that first in *both* cases.
    user_host = os.getenv("REDIS_HOST") or host

    if in_docker:
        # Inside containers, 'redis' is the Docker service DNS name.
        candidates = [
            user_host,
            "redis",
            "host.docker.internal",  # often works on Mac/Win
            "127.0.0.1",
            "localhost",
        ]
    else:
        # On the host OS, Redis is usually published on localhost.
        candidates = [
            user_host,
            "127.0.0.1",
            "localhost",
            "redis",  # just in case someone runs the notebook in another container
        ]

    candidates = _uniq(candidates)

    last_err = None
    print(f"[INFO] Running in Docker: {in_docker}. Trying Redis hosts {candidates} on port {resolved_port}.")

    for cand in candidates:
        for attempt in range(1, tries + 1):
            try:
                # DNS resolution first (clearer errors)
                socket.getaddrinfo(cand, resolved_port)

                r = redis.Redis(
                    host=cand,
                    port=resolved_port,
                    db=0,
                    password=resolved_pwd,
                    socket_connect_timeout=per_try_timeout,
                    socket_timeout=per_try_timeout,
                    retry_on_timeout=True,
                )
                r.ping()
                print(f"[OK] Connected to Redis at {cand}:{resolved_port} (attempt {attempt}).")
                return r
            except Exception as e:
                last_err = e
                print(f"[WARN] Connect failed to {cand}:{resolved_port} (attempt {attempt}/{tries}): {e}")
                time.sleep(backoff)

        print(f"[INFO] Switching to next candidate host after {tries} failed attempts: {cand}")

    # If we reach here, all candidates failed
    hint = (
        "If you are running *inside* the Spark container, make sure Redis is started with Docker and the service is named 'redis'.\n"
        "  -> Run: docker compose up -d\n\n"
        "If you are running on your Mac (outside Docker), make sure Redis is published on localhost:6379 by your compose file.\n"
        "You can also set an explicit REDIS_HOST, e.g.:\n"
        "  export REDIS_HOST=redis      # (inside container)\n"
        "  export REDIS_HOST=127.0.0.1  # (on host)\n"
    )
    raise ConnectionError(
        f"❌ Could not reach Redis via {candidates} on port {resolved_port}.\nLast error: {last_err}\n\n{hint}"
    )

# ---- Use it ----
r = connect_redis_smart()

# Quick sanity check (won't fail if empty)
try:
    sample_keys = r.keys("city:*:window:*")
    print(f"[INFO] Sample window keys found: {len(sample_keys)}")
    if sample_keys[:3]:
        print("  e.g.", [k.decode() if isinstance(k, bytes) else k for k in sample_keys[:3]])
except Exception as e:
    print(f"[WARN] Connected but key listing failed (likely fine if pipeline not started yet): {e}")


[INFO] Running in Docker: False. Trying Redis hosts ['127.0.0.1', 'localhost', 'redis'] on port 6379.
[OK] Connected to Redis at 127.0.0.1:6379 (attempt 1).
[INFO] Sample window keys found: 1130
  e.g. ['city:StPete:window:202511091600', 'city:Orlando:window:202511082253', 'city:Jacksonville:window:202511082359']


/var/folders/xb/g_pcys_x2dzfp2c67z19s1tm0000gn/T/ipykernel_12326/1877276641.py:101: DeprecationWarning:

Call to '__init__' function with deprecated usage of input argument/s 'retry_on_timeout'. (TimeoutError is included by default.) -- Deprecated since version 6.0.0.



## Fetch Minute Window Data

In [91]:
def iter_minute_keys(city, start, end):
    t = start
    while t < end:
        yield f"city:{city}:window:{t.strftime('%Y%m%d%H%M')}", t
        t += timedelta(minutes=1)

def fetch_window_df(last_minutes=30):
    now_utc = datetime.now(timezone.utc)
    start_utc = now_utc - timedelta(minutes=last_minutes)
    rows = []

    for city in CITIES:
        for key, key_dt in iter_minute_keys(city, start_utc, now_utc):
            data = r.hgetall(key)
            if not data:
                continue
            d = {k.decode(): v.decode() for k, v in data.items()}
            rows.append({
                "city": city,
                "window_start": datetime.fromisoformat(d["window_start"]).astimezone(DISPLAY_TZ),
                "window_end":   datetime.fromisoformat(d["window_end"]).astimezone(DISPLAY_TZ),
                "total_watts":  float(d["total_watts"]),
                "avg_watts":    float(d["avg_watts"]),
                "max_watts":    float(d["max_watts"]),
                "count":        int(float(d["count"])),
            })

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["window_start","city"]).reset_index(drop=True)


## Fetch Recent Alerts

In [92]:
def fetch_recent_alerts(limit=200):
    rows = []
    for city in CITIES:
        keys = r.lrange(f"alerts:{city}", 0, limit)
        for ak in keys:
            ak = ak.decode()
            d = {k.decode(): v.decode() for k, v in r.hgetall(ak).items()}
            rows.append({
                "city": d["city"],
                "anomaly_type": d["anomaly_type"],
                "total_watts": float(d["total_watts"]),
                "avg_watts": float(d["avg_watts"]),
                "max_watts": float(d["max_watts"]),
                "window_start": datetime.fromisoformat(d["window_start"]).astimezone(DISPLAY_TZ),
                "window_end": datetime.fromisoformat(d["window_end"]).astimezone(DISPLAY_TZ),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["window_start","city"]).reset_index(drop=True)


## Total Watts Timeseries (Plotly)

In [93]:
HISTORY_MINUTES = 30
df = fetch_window_df(HISTORY_MINUTES)
if df.empty:
    print("No data yet. Start producer + stream job.")
else:
    fig = px.line(
        df, x="window_start", y="total_watts",
        color="city", markers=True,
        title=f"City Total Watts – Last {HISTORY_MINUTES} Minutes"
    )
    fig.update_layout(hovermode="x unified")
    fig.show()


## Avg vs Max Watts Comparison

In [94]:
if not df.empty:
    df_long = df.melt(
        id_vars=["city","window_start"],
        value_vars=["avg_watts","max_watts"],
        var_name="metric", value_name="watts"
    )
    fig = px.line(
        df_long, x="window_start", y="watts",
        color="city", line_dash="metric",
        title="City Avg vs Max Watts"
    )
    fig.update_layout(hovermode="x unified")
    fig.show()


## Latest Window Snapshot (Grouped Bars)

In [95]:
if not df.empty:
    latest = df.loc[df.groupby("city")["window_start"].idxmax()]
    fig = go.Figure()
    fig.add_bar(x=latest["city"], y=latest["total_watts"], name="Total")
    fig.add_bar(x=latest["city"], y=latest["avg_watts"], name="Avg")
    fig.add_bar(x=latest["city"], y=latest["max_watts"], name="Max")
    fig.update_layout(barmode="group", title="Latest Completed Window per City")
    fig.show()


## Anomaly view by city(last 15 minutes)

In [96]:
# ---- Simple anomaly view by city (last 15 minutes) ----
from datetime import datetime, timezone, timedelta
import pandas as pd

alerts = fetch_recent_alerts()

if alerts is None or alerts.empty:
    print("No anomalies detected in the last 15 minutes.")
else:
    # Ensure timestamps are timezone-aware
    for col in ("window_start", "window_end"):
        if col in alerts.columns:
            alerts[col] = pd.to_datetime(alerts[col], utc=True, errors="coerce")

    now_utc = datetime.now(timezone.utc)
    cutoff = now_utc - timedelta(minutes=15)

    # Keep alerts overlapping last 15 minutes
    recent = alerts[alerts["window_end"] >= cutoff].copy()

    if recent.empty:
        print("No anomalies detected in the last 15 minutes.")
    else:
        # Clean display columns
        display_cols = [c for c in [
            "city", "anomaly_type", "window_start", "window_end",
            "total_watts", "avg_watts", "max_watts", "reason"
        ] if c in recent.columns]

        recent = recent.sort_values(["city", "window_end"], ascending=[True, False])

        print(f"Anomalies in the Last 15 Minutes: {len(recent)}")
        display(recent[display_cols].reset_index(drop=True))

        # ---- Counts per city ----
        if "city" in recent.columns:
            print("\nAnomaly Count by City (last 15 min):")
            city_counts = recent["city"].value_counts()
            for city, count in city_counts.items():
                print(f"  - {city}: {count}")

        # ---- Counts by type per city ----
        if {"city", "anomaly_type"}.issubset(recent.columns):
            print("\nAnomaly Type Breakdown per City:")
            pivot = recent.groupby(["city", "anomaly_type"]).size().unstack(fill_value=0)
            display(pivot)


Anomalies in the Last 15 Minutes: 353


,city,anomaly_type,window_start,window_end,total_watts,avg_watts,max_watts
0,Jacksonville,HIGH_TOTAL,2025-11-09 21:17:00+00:00,2025-11-09 21:18:00+00:00,67594.59,1228.992545,2159.45
1,Jacksonville,SPIKE,2025-11-09 21:16:00+00:00,2025-11-09 21:17:00+00:00,64058.20,1231.888462,2187.52
2,Jacksonville,SPIKE,2025-11-09 21:15:00+00:00,2025-11-09 21:16:00+00:00,70131.74,1298.735926,2196.21
3,Jacksonville,HIGH_TOTAL,2025-11-09 21:15:00+00:00,2025-11-09 21:16:00+00:00,70131.74,1298.735926,2196.21
4,Jacksonville,HIGH_TOTAL,2025-11-09 21:14:00+00:00,2025-11-09 21:15:00+00:00,65280.49,1208.897963,2171.99
...,...,...,...,...,...,...,...
348,Tampa,SPIKE,2025-11-09 20:15:00+00:00,2025-11-09 20:16:00+00:00,74252.19,1350.039818,2187.00
349,Tampa,HIGH_TOTAL,2025-11-09 20:15:00+00:00,2025-11-09 20:16:00+00:00,74252.19,1350.039818,2187.00
350,Tampa,HIGH_TOTAL,2025-11-09 20:14:00+00:00,2025-11-09 20:15:00+00:00,68699.32,1272.209630,2179.48
351,Tampa,HIGH_TOTAL,2025-11-09 20:13:00+00:00,2025-11-09 20:14:00+00:00,70512.63,1282.047818,2150.17



Anomaly Count by City (last 15 min):
  - Miami: 81
  - StPete: 74
  - Jacksonville: 70
  - Orlando: 65
  - Tampa: 63

Anomaly Type Breakdown per City:


anomaly_type,HIGH_AVG,HIGH_TOTAL,SPIKE
city,,,
Jacksonville,1,38,31
Miami,13,41,27
Orlando,5,36,24
StPete,5,40,29
Tampa,6,33,24


## Interactive City Selector

In [97]:
import ipywidgets as widgets
from IPython.display import display

if not df.empty:
    dropdown = widgets.Dropdown(options=CITIES, description="City:")
    out = widgets.Output()

    def render(city):
        out.clear_output()
        with out:
            d = df[df.city == city]
            fig = px.line(d, x="window_start", y=["total_watts","avg_watts","max_watts"],
                          title=f"{city} – Usage Breakdown", markers=True)
            fig.show()

    dropdown.observe(lambda c: render(c["new"]) if c["name"]=="value" else None)
    display(dropdown, out)
    render(dropdown.value)


Dropdown(description='City:', options=('Tampa', 'Orlando', 'Miami', 'StPete', 'Jacksonville'), value='Tampa')

Output()